# SAND → CSV (otoole) converter

Convierte un archivo en formato **SAND** (una sola hoja con todos los parámetros apilados,
años en columnas) al formato **CSV de otoole** (un archivo por parámetro/set, formato largo
con columna `VALUE`).

**Lógica:**
1. Se lee `config_depurado.yaml` (la misma configuración de otoole) como fuente de verdad
   de los índices de cada parámetro.
2. Para cada parámetro presente en el SAND se filtran sus filas y se conservan **solo** las
   columnas de índice que ese parámetro usa (las demás quedan vacías en el SAND y se descartan).
3. Si el parámetro está indexado por `YEAR`, las columnas de años se despivotan (`melt`)
   a formato largo. Si no, el valor se toma de la columna `Time indipendent variables`.
4. Los SETs (`TECHNOLOGY`, `FUEL`, `YEAR`, ...) se derivan de los valores únicos del archivo.
5. Los parámetros del config que no aparecen en el SAND se escriben como CSV vacíos con
   sus encabezados correctos (otoole los requiere y usará su valor `default`).

El resultado es directamente utilizable con `otoole convert csv excel ...`.

In [1]:
from pathlib import Path

import pandas as pd
import yaml

# ------------------------- Configuración -------------------------
SAND_FILE = Path("SAND_Regional/SAND_CN_Regionalizado.xlsx")   # archivo SAND de entrada
CONFIG_FILE = Path("config_depurado.yaml")             # config de otoole (índices por parámetro)
OUTPUT_DIR = Path("CSV_Regional_CN/")                    # carpeta de salida (se crea si no existe)

SHEET_NAME = 0  # primera hoja ("Parameters" en el SAND base, "Hoja1" en los reducidos)
TIME_INDEP_COL = "Time indipendent variables"          # columna de valores sin índice YEAR

In [2]:
# ------------------- Cargar config de otoole ---------------------
with open(CONFIG_FILE) as f:
    config = yaml.safe_load(f)

params = {name: spec for name, spec in config.items() if spec["type"] == "param"}
sets = {name: spec for name, spec in config.items() if spec["type"] == "set"}

print(f"{len(params)} parámetros y {len(sets)} sets definidos en el config")

40 parámetros y 8 sets definidos en el config


In [3]:
# ----------------------- Leer archivo SAND -----------------------
sand = pd.read_excel(SAND_FILE, sheet_name=SHEET_NAME)
sand.columns = [str(c).strip() for c in sand.columns]

# Las columnas de año son las que tienen encabezado numérico (2022, 2023, ...)
year_cols = [c for c in sand.columns if c.isdigit()]

print(f"{len(sand)} filas | años {year_cols[0]}–{year_cols[-1]}")
print(f"Parámetros presentes en el SAND: {sand['Parameter'].nunique()}")

# Avisar si el SAND trae parámetros que el config no conoce (no se convertirían)
unknown = set(sand["Parameter"].unique()) - set(params)
if unknown:
    print("¡ATENCIÓN! Parámetros en el SAND que NO están en el config:", sorted(unknown))

35797 filas | años 2022–2054
Parámetros presentes en el SAND: 32


In [4]:
# --------------------- Funciones de conversión -------------------
def cast_index(series: pd.Series, index_name: str) -> pd.Series:
    """Aplica el dtype que el config define para cada set/índice."""
    if sets[index_name]["dtype"] == "int":
        return series.astype(float).astype(int)   # evita "1.0" en índices enteros
    return series.astype(str)


def extract_parameter(name: str, spec: dict) -> pd.DataFrame:
    """Extrae un parámetro del SAND y lo devuelve en formato largo de otoole."""
    indices = spec["indices"]
    rows = sand.loc[sand["Parameter"] == name]

    if "YEAR" in indices:
        # Despivotar las columnas de año a formato largo
        id_cols = [i for i in indices if i != "YEAR"]
        df = rows.melt(id_vars=id_cols, value_vars=year_cols,
                       var_name="YEAR", value_name="VALUE")
    else:
        # Parámetro sin índice YEAR: el valor está en la columna de tiempo-independiente
        df = rows[indices].copy()
        df["VALUE"] = rows[TIME_INDEP_COL].values

    df = df.dropna(subset=["VALUE"])               # celdas vacías = usar default de otoole
    for idx in indices:
        df[idx] = cast_index(df[idx], idx)
    df["VALUE"] = df["VALUE"].astype(spec["dtype"])

    return df[indices + ["VALUE"]].sort_values(indices).reset_index(drop=True)

In [5]:
# ------------------ Convertir todos los parámetros ---------------
OUTPUT_DIR.mkdir(exist_ok=True)
summary = []

for name, spec in params.items():
    df = extract_parameter(name, spec)
    df.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
    summary.append({"file": f"{name}.csv", "type": "param", "rows": len(df)})

In [6]:
# ------------------------ Derivar los SETs -----------------------
for set_name, spec in sets.items():
    if set_name == "YEAR":
        values = sorted(int(y) for y in year_cols)
    elif set_name in sand.columns:
        col = sand[set_name].dropna()
        if spec["dtype"] == "int":
            values = sorted(col.astype(float).astype(int).unique())
        else:
            values = sorted(col.astype(str).unique())
    else:
        values = []  # set sin columna en el SAND (p. ej. STORAGE si no se usa)

    pd.DataFrame({"VALUE": values}).to_csv(OUTPUT_DIR / f"{set_name}.csv", index=False)
    summary.append({"file": f"{set_name}.csv", "type": "set", "rows": len(values)})

In [7]:
# --------------------------- Resumen -----------------------------
report = pd.DataFrame(summary).sort_values(["type", "file"]).reset_index(drop=True)
empty = report[(report["type"] == "param") & (report["rows"] == 0)]

print(f"{len(report)} archivos escritos en {OUTPUT_DIR.resolve()}")
print(f"{len(empty)} parámetros sin datos en el SAND (CSV vacío, otoole usará el default):")
print("   " + ", ".join(empty["file"].str.replace(".csv", "", regex=False)))
report

48 archivos escritos en C:\Users\carlo\Documents\05_Gestion_Empresas\01_UPME\Proyectos\Generar_Escenario_Regional\CSV_Regional_CN
8 parámetros sin datos en el SAND (CSV vacío, otoole usará el default):
   DiscountRateIdv, EmissionsPenalty, REMinProductionTarget, RETagFuel, ReserveMargin, ReserveMarginTagFuel, TechnologyFromStorage, TechnologyToStorage


,file,type,rows
0,AccumulatedAnnualDemand.csv,param,9636
1,AnnualEmissionLimit.csv,param,99
2,AnnualExogenousEmission.csv,param,99
3,AvailabilityFactor.csv,param,5049
4,CapacityFactor.csv,param,3597
5,CapacityOfOneTechnologyUnit.csv,param,3036
6,CapacityToActivityUnit.csv,param,2371
7,CapitalCost.csv,param,75339
8,DepreciationMethod.csv,param,1
9,DiscountRate.csv,param,1


## Siguiente paso

La carpeta de salida ya es un directorio CSV válido de otoole. Para obtener el
Excel estándar (una hoja por parámetro, años pivotados):

```bash
otoole convert csv excel CSV_desde_SAND salida.xlsx config_depurado.yaml
```

**Para convertir otro SAND** (p. ej. los de `SANDs_Reducidos/`), basta con cambiar
`SAND_FILE` y `OUTPUT_DIR` en la celda de configuración y volver a ejecutar. Nota:
los SAND *reducidos* solo contienen un subconjunto de parámetros — los demás saldrán
como CSV vacíos, lo cual es el comportamiento esperado.